### PyTorch ResNet Exercises

Welcome to the PyTorch ResNet exercise template notebook.

There are several questions in this notebook and it's your goal to answer them by writing Python and PyTorch code.






In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader
import numpy as np
import time
import requests
from zipfile import ZipFile
from io import BytesIO

# Define the ResNet architecture
class ResNet(nn.Module):
    def __init__(self, num_classes=200):
        super(ResNet, self).__init__()
        self.resnet = models.resnet18(pretrained=False)  # You can use other ResNet variants like resnet50 if needed
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)


# Hyperparameters
learning_rates = [0.001]
batch_sizes = [32]

# Initialize the network
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
net = ResNet(num_classes=200).to(device)

# Load the Tiny ImageNet dataset
# Note: You'll need to download the dataset and set the correct path.
dataset_path = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'  # Replace with the path to your dataset


# Send a GET request to the URL
response = requests.get(dataset_path)
# Check if the request was successful
if response.status_code == 200:
    # Open the downloaded bytes and extract them
    with ZipFile(BytesIO(response.content)) as zip_file:
        zip_file.extractall('/dataset')
    print('Download and extraction complete!')

# Define transforms for the input data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


train_dataset = datasets.ImageFolder(root='/dataset/tiny-imagenet-200/train', transform=transform)
val_dataset = datasets.ImageFolder(root='/dataset/tiny-imagenet-200/val', transform=transform)

# Loop over hyperparameters
for lr in learning_rates:
    for batch_size in batch_sizes:
        # Data loaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

        # Train the network
        num_epochs = 10
        for epoch in range(num_epochs):
            # Training loop
            net.train()
            running_loss = 0.0
            for images, labels in train_loader:
                optimizer.zero_grad()
                images = images.to(device)
                labels = labels.to(device)
                outputs = net(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * images.size(0)

            # Validation loop
            net.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images = images.to(device)
                    labels = labels.to(device)
                    outputs = net(images)
                    _, predicted = torch.max(outputs, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()
            val_accuracy = correct / total

            # Evaluate on the validation set and print results
            print(f'LR: {lr}, Batch Size: {batch_size}, Epoch: {epoch+1}, Validation Accuracy: {val_accuracy:.4f}')

            # You can save the model state if you want to keep it
            torch.save(net.state_dict(), f'resnet_lr{lr}_bs{batch_size}.pth')


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Download and extraction complete!
LR: 0.001, Batch Size: 32, Epoch: 1, Validation Accuracy: 0.0035
LR: 0.001, Batch Size: 32, Epoch: 2, Validation Accuracy: 0.0050
LR: 0.001, Batch Size: 32, Epoch: 3, Validation Accuracy: 0.0106
LR: 0.001, Batch Size: 32, Epoch: 4, Validation Accuracy: 0.0047
